# Cleaning4
Livello di pulizia che prende i file cleaned_01 fa le elaborazioni di cleaning 2 & 3 ma senza eliminare righe.\
File così creati servono per il merge.
## Inizializzazione ed Import

In [10]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [11]:
file_codes = ['BAIPETNMRCFTP',
'AMPRION_ASYN_SAA',
'PLASMA_ABETA_PROJECT_ADX_VUMC',
'FNIH_PLASMA_PTAU_PROJECT',
'CSFALPHASYN',
'BLENNOWCSFNFL',
'UPENNPLASMA',
'BLENNOWPLASMANFL',
'BLENNOWPLASMATAU',
'C2N_PRECIVITYAD2_PLASMA',
'UGOTPTAU181',
'UPENN_PLASMA_FUJIREBIO_QUANTERIX',
'UCBERKELEY_AMY_6MM',
'UCBERKELEY_TAUPVC_6MM']


In [12]:
search = client.query_files(
    query={'custom.level' : 'cleaned_01', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [13]:
len(zip_files)

13

## Operazioni
- Trasformare in dummies alcuni parametri
- Normalizzare i volumi
- nuovi metadati (cofattori e fattori)

In [14]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned_BB'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned4BB'

if os.path.isfile(new_name+'.xlsx'):
    #aggiunge i filecode mancanti e riporta i file_code da riprocessare allo status precedente (variable names)
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)


The ADNI_variables_cleaned4BB file has been updated with the new file_code: []
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned4BB file has restored the previous information of the file_code: []
Open the file and verify it, if needed update the variables names and metadata


In [15]:
new_support_file = pd.read_excel(new_name+'.xlsx')
dataCleaner = DataCleaner(support_file=new_support_file)

In [16]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    new_support_file = dataCleaner.update_self_support_file(new_support_file)
    
    file_code, metadat_custom = dataCleaner.get_file_code_metadata(file_name, prefix='cleaned/single_file')

    processed_df = df_new.copy(deep=True)

    # funzione che trasforma parametri categorici in dummies
    ref_list = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
    to_dummy_list = [x for x in ref_list if x in processed_df.columns]
    if to_dummy_list:
        print('\n\n Dummies in ----', file_name)
        processed_df, bool_var = dataCleaner.classes_to_dummies(processed_df, col_list=to_dummy_list) 
    
    if 'volume' in support_file[support_file['file_code'] == file_code]['metadati_normalizzazione'].values:
        print('\n\n Volumes in ----', file_name)
        # verifica che i volumi siano già totali e non solo una parte laterale
        processed_df, volume_list = dataCleaner.get_volumes_total(processed_df, file_code)
        print(processed_df.columns)
        # Transform volumes as ICV percentage
        processed_df = dataCleaner.transform_volumes_as_ICV_percent(processed_df, volume_list, file_code)
    
    final_df = processed_df.copy(deep=True)
    
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_04')

    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
    
    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_04', file_name=file_name, prefix='cleaned/single_file', updated_support_file=new_support_file)    
    
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)
            new_support_file = infoSupportFile.get_subjects_and_multiplevisits(key)
    
    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name) 



 ---- BAIPETNMRCFTP_08_17_22_11Aug2025_01.csv


 ---- AMPRION_ASYN_SAA_11Aug2025_01.csv


 ---- PLASMA_ABETA_PROJECT_ADX_VUMC_11Aug2025_01.csv


 ---- FNIH_PLASMA_PTAU_PROJECT_11Aug2025_01.csv


 ---- CSFALPHASYN_03_21_14_11Aug2025_01.csv


 ---- BLENNOWCSFNFL_11Aug2025_01.csv


 ---- ADNI_BLENNOWPLASMANFLLONG_10_03_18_11Aug2025_01.csv


 ---- BLENNOWPLASMATAU_11Aug2025_01.csv


 ---- C2N_PRECIVITYAD2_PLASMA_11Aug2025_01.csv


 ---- UGOTPTAU181_06_18_20_28Oct2025_01.csv


 ---- UPENN_PLASMA_FUJIREBIO_QUANTERIX_28Oct2025_01.csv


 ---- UCBERKELEY_AMY_6MM_28Oct2025_01.csv


 ---- UCBERKELEY_TAUPVC_6MM_28Oct2025_01.csv


In [17]:
final_df

,LONIUID,PTID,RID,VISCODE2,SCANDATE,SITEID,PROCESSDATE,TRACER,TRACER_SUVR_WARNING,META_TEMPORAL_SUVR,...,RIGHT_PUTAMEN_VOLUME,RIGHT_THALAMUS_PROPER_SUVR,RIGHT_THALAMUS_PROPER_VOLUME,RIGHT_VENTRALDC_SUVR,RIGHT_VENTRALDC_VOLUME,RIGHT_VESSEL_SUVR,RIGHT_VESSEL_VOLUME,Apositive,Tpositive,Npositive
0,1594604,011_S_0021,21,m144,2018-02-02,11,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.239,...,4225.0,1.447,5781.0,1.399,3422.0,1.315,14.0,1.0,1.0,0.0
1,1596177,023_S_0031,31,m150,2018-04-24,23,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.167,...,5712.0,1.197,6261.0,1.325,2959.0,1.547,6.0,1.0,1.0,0.0
2,1596172,023_S_0031,31,m162,2019-04-23,23,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.139,...,5967.0,1.134,5947.0,1.241,3008.0,1.438,8.0,1.0,1.0,0.0
3,1598898,067_S_0056,56,m144,2018-02-20,67,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.275,...,4024.0,1.398,6595.0,1.431,3646.0,1.584,42.0,1.0,1.0,0.0
4,1598985,067_S_0056,56,m156,2019-01-10,67,2022-09-02,FTP,DO NOT COMPARE SUVRs ACROSS TRACERS,1.275,...,4155.0,1.455,6386.0,1.474,3681.0,1.433,39.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2310,11342478,019_S_10705,10705,bl,2025-06-30,19,2025-08-19,PI2620,DO NOT COMPARE SUVRs ACROSS TRACERS,1.432,...,4503.0,0.959,7596.0,1.194,4239.0,0.963,6.0,1.0,0.0,0.0
2311,11427984,031_S_10720,10720,bl,2025-08-15,31,2025-09-11,PI2620,DO NOT COMPARE SUVRs ACROSS TRACERS,1.146,...,3491.0,1.026,5722.0,1.100,3348.0,0.979,1.0,0.0,0.0,0.0
2312,11428493,031_S_10731,10731,bl,2025-08-18,31,2025-09-11,PI2620,DO NOT COMPARE SUVRs ACROSS TRACERS,1.083,...,5140.0,0.968,7676.0,1.075,4505.0,0.933,3.0,0.0,0.0,0.0
2313,11346392,019_S_10802,10802,bl,2025-06-30,19,2025-07-29,PI2620,DO NOT COMPARE SUVRs ACROSS TRACERS,1.214,...,4523.0,0.973,6503.0,1.363,3621.0,0.877,46.0,1.0,0.0,0.0


In [18]:
updated_metadata

{'file_code': 'UCBERKELEY_TAUPVC_6MM',
 'level': 'cleaned_04',
 'population': ['ADNI4', 'ADNI3', 'ADNI2'],
 'source': 'ADNI',
 'cofattori': [],
 'predittori': ['CSF_SUVR',
  'CTX_ENTORHINAL_VOLUME',
  'CSF_VOLUME',
  'CTX_FUSIFORM_VOLUME',
  'CTX_INFERIORPARIETAL_VOLUME',
  'CTX_INFERIORTEMPORAL_VOLUME',
  'CTX_LATERALOCCIPITAL_VOLUME',
  'CTX_MIDDLETEMPORAL_VOLUME',
  'CTX_PARAHIPPOCAMPAL_VOLUME',
  'CTX_PRECUNEUS_VOLUME'],
 'norm_scala': [],
 'norm_intervallo': [],
 'norm_volume': [],
 'norm_scale_value': [],
 'volume_norm_values': []}